# Étape 3 — Entraînement et comparaison de 3 modèles CNN
## Projet : Classification automatique de déchets (TrashNet)

Nous allons comparer 3 approches :
1. **CNN Simple** — réseau de base (référence)
2. **CNN Profond** — avec BatchNorm + Dropout + Data Augmentation *(corrigé)*
3. **Transfer Learning** — MobileNetV2 pré-entraîné + **fine-tuning** *(amélioré)*




## 0. Installation et imports

In [ ]:
%pip install tensorflow kagglehub matplotlib scikit-learn seaborn numpy pandas pillow

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight  # ✅ AJOUT

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version :', tf.__version__)
print('Imports OK !')

---
## 1. Téléchargement et chargement des données

In [ ]:
import kagglehub

path = kagglehub.dataset_download("feyzazkefe/trashnet")
BASE_DIR = os.path.join(path, "dataset-resized")

print("Dataset téléchargé :", BASE_DIR)

In [ ]:
# ✅ CORRECTION : résolution 224×224 (taille native de MobileNetV2)
# Avant : IMG_SIZE = (128, 128) → sous-optimal pour MobileNetV2
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 20   # ✅ Augmenté car EarlyStopping arrêtera au bon moment
CATEGORIES  = sorted([
    c for c in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, c))
])
NUM_CLASSES = len(CATEGORIES)
label_map   = {cat: i for i, cat in enumerate(CATEGORIES)}

print(f"Catégories ({NUM_CLASSES}) :", CATEGORIES)
print("Label map :", label_map)
print(f"Résolution d'entrée : {IMG_SIZE}")

In [ ]:
# Chargement de toutes les images en mémoire
X, y = [], []

for cat in CATEGORIES:
    folder = os.path.join(BASE_DIR, cat)
    for img_name in os.listdir(folder):
        path_img = os.path.join(folder, img_name)
        try:
            img = Image.open(path_img).convert('RGB').resize(IMG_SIZE)
            X.append(np.array(img) / 255.0)
            y.append(label_map[cat])
        except Exception:
            pass

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Total images : {len(X)}")

In [ ]:
# Afficher 1 exemple par catégorie
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(16, 3))
for i, cat in enumerate(CATEGORIES):
    idx = np.where(y == i)[0][0]
    axes[i].imshow(X[idx])
    axes[i].set_title(cat, fontsize=9, fontweight='bold')
    axes[i].axis('off')
plt.suptitle("Exemples par catégorie", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exemples_categories.png', dpi=120)
plt.show()

In [ ]:
# Distribution des classes
counts = pd.Series(y).map({v: k for k, v in label_map.items()}).value_counts()
plt.figure(figsize=(8, 4))
bars = plt.bar(counts.index, counts.values,
               color=['#2E8B57','#4169E1','#DC143C','#FF8C00','#9370DB','#20B2AA'])
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha='center', fontsize=10, fontweight='bold')
plt.title("Distribution des images par catégorie", fontsize=13, fontweight='bold')
plt.xlabel("Catégorie")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('distribution_categories.png', dpi=120)
plt.show()

---
## 2. Séparation des données (train / validation / test)

In [ ]:
# Encodage one-hot des labels
y_onehot = keras.utils.to_categorical(y, NUM_CLASSES)

# Split : 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp, y_int_train, y_int_temp = train_test_split(
    X, y_onehot, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Train      : {X_train.shape[0]} images")
print(f"Validation : {X_val.shape[0]} images")
print(f"Test       : {X_test.shape[0]} images")

# ✅ AJOUT : Class weights pour compenser le déséquilibre (trash sous-représenté)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_int_train),
    y=y_int_train
)
class_weight_dict = dict(enumerate(class_weights))
print("\nClass weights calculés :")
for cat, w in zip(CATEGORIES, class_weights):
    print(f"  {cat:<12} : {w:.3f}")

---
## 3. Callbacks communs (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)
> ✅ AJOUT : Ces callbacks évitent l'overfitting et sauvegardent le meilleur modèle automatiquement

In [ ]:
def get_callbacks(model_name):
    """Retourne les callbacks standards pour l'entraînement"""
    return [
        # Arrêt automatique si val_accuracy ne s'améliore plus pendant 5 époques
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=5,
            restore_best_weights=True,  # recharge les meilleurs poids
            verbose=1
        ),
        # Divise le learning rate par 2 si val_loss stagne pendant 3 époques
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        # Sauvegarde le meilleur modèle
        keras.callbacks.ModelCheckpoint(
            filepath=f'best_{model_name}.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=0
        )
    ]

print("Callbacks définis !")

---
## 4. Modèle 1 — CNN Simple (référence)
> Réseau basique : 2 couches Conv2D + MaxPooling + Dense

In [ ]:
def build_cnn_simple():
    model = keras.Sequential([
        # Bloc 1
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
        layers.MaxPooling2D(2,2),

        # Bloc 2
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        # Classifieur
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_Simple')

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model1 = build_cnn_simple()
model1.summary()

In [ ]:
print("Entraînement du CNN Simple...")
history1 = model1.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict,  # ✅ AJOUT
    callbacks=get_callbacks('cnn_simple'),  # ✅ AJOUT
    verbose=1
)

model1.save('model_cnn_simple.h5')
print("Modèle 1 sauvegardé : model_cnn_simple.h5")

---
## 5. Modèle 2 — CNN Profond + Dropout + Data Augmentation


In [ ]:
# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)
datagen.fit(X_train)
print("Data Augmentation configurée !")

In [ ]:
# Afficher des exemples d'images augmentées
sample = X_train[:1]
fig, axes = plt.subplots(1, 6, figsize=(14, 3))
axes[0].imshow(sample[0])
axes[0].set_title('Original')
axes[0].axis('off')
for i, batch in enumerate(datagen.flow(sample, batch_size=1)):
    if i >= 5: break
    axes[i+1].imshow(batch[0])
    axes[i+1].set_title(f'Aug {i+1}')
    axes[i+1].axis('off')
plt.suptitle("Exemples de Data Augmentation", fontweight='bold')
plt.tight_layout()
plt.savefig('data_augmentation.png', dpi=120)
plt.show()

In [ ]:
# ✅ CORRECTION : learning_rate réduit à 5e-4 (au lieu de 1e-3)
# Raison : avec Data Augmentation, un LR plus faible stabilise l'entraînement
def build_cnn_profond():
    model = keras.Sequential([
        # Bloc 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(224,224,3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Bloc 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Bloc 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # ✅ AJOUT : Bloc 4 supplémentaire
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),  # ✅ Plus robuste que Flatten pour éviter l'overfit

        # Classifieur
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_Profond')

    # ✅ CORRECTION : LR réduit de 1e-3 à 5e-4
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model2 = build_cnn_profond()
model2.summary()

In [ ]:
print("Entraînement du CNN Profond...")
history2 = model2.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    class_weight=class_weight_dict,  # ✅ AJOUT
    callbacks=get_callbacks('cnn_profond'),  # ✅ AJOUT
    verbose=1
)

model2.save('model_cnn_profond.h5')
print("Modèle 2 sauvegardé : model_cnn_profond.h5")

---
## 6. Modèle 3 — Transfer Learning (MobileNetV2) + Fine-tuning


In [ ]:
def build_transfer_learning():
    # ✅ CORRECTION : input_shape=(224,224,3) — taille native de MobileNetV2
    # Avant : (128,128,3) → perte d'information significative
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )

    # Phase 1 : geler toutes les couches de base
    base_model.trainable = False

    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),  # ✅ Augmenté de 128 à 256
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='Transfer_Learning_MobileNetV2')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model, base_model

model3, base_model = build_transfer_learning()
model3.summary()

In [ ]:
# ── Phase 1 : entraîner uniquement la tête de classification ──
print("Phase 1 : Entraînement de la tête (base gelée)...")
history3_phase1 = model3.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    epochs=10,
    validation_data=(X_val, y_val),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    class_weight=class_weight_dict,  # ✅ AJOUT
    callbacks=get_callbacks('tl_phase1'),
    verbose=1
)
print("Phase 1 terminée !")

In [ ]:
# ── Phase 2 : Fine-tuning des 30 dernières couches ──
# ✅ AJOUT : dégeler les 30 dernières couches pour affiner les features
print("Phase 2 : Fine-tuning des 30 dernières couches...")

base_model.trainable = True
# Geler toutes les couches SAUF les 30 dernières
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f"Couches entraînables : {sum(l.trainable for l in model3.layers[0].layers)} / {len(model3.layers[0].layers)}")

# ✅ LR très faible pour ne pas détruire les poids pré-entraînés
model3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history3_phase2 = model3.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    epochs=15,
    validation_data=(X_val, y_val),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    class_weight=class_weight_dict,  # ✅ AJOUT
    callbacks=get_callbacks('tl_phase2'),
    verbose=1
)

model3.save('model_transfer_learning.h5')
print("Modèle 3 sauvegardé : model_transfer_learning.h5")

In [ ]:
# ✅ Fusionner les deux historiques pour l'affichage
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history3_merged = merge_histories(history3_phase1, history3_phase2)
print(f"Historique fusionné : {len(history3_merged['accuracy'])} époques au total")

---
## 7. Évaluation et comparaison des 3 modèles

In [ ]:
# Accuracy sur le jeu de test
_, acc1 = model1.evaluate(X_test, y_test, verbose=0)
_, acc2 = model2.evaluate(X_test, y_test, verbose=0)
_, acc3 = model3.evaluate(X_test, y_test, verbose=0)

resultats = pd.DataFrame({
    'Modèle'    : ['CNN Simple', 'CNN Profond + Augmentation', 'Transfer Learning (MobileNetV2)'],
    'Accuracy'  : [f'{acc1*100:.1f}%', f'{acc2*100:.1f}%', f'{acc3*100:.1f}%'],
    'Complexité': ['Faible', 'Moyenne', 'Élevée']
})
print(resultats.to_string(index=False))

In [ ]:
# Graphique comparatif des accuracy
modeles  = ['CNN\nSimple', 'CNN Profond\n+ Augmentation', 'Transfer\nLearning']
accuracy = [acc1*100, acc2*100, acc3*100]
colors   = ['#4169E1', '#2E8B57', '#DC143C']

plt.figure(figsize=(8, 5))
bars = plt.bar(modeles, accuracy, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, accuracy):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
plt.ylim(0, 110)
plt.title('Comparaison des 3 modèles — Accuracy sur le jeu de test',
          fontsize=13, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=120)
plt.show()

In [ ]:
# Courbes d'apprentissage des 3 modèles
# ✅ Pour le TL, on utilise l'historique fusionné Phase1 + Phase2
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

histories_data = [
    (history1.history, 'CNN Simple',       '#4169E1'),
    (history2.history, 'CNN Profond',      '#2E8B57'),
    (history3_merged,  'Transfer Learning','#DC143C'),
]

for ax, (hist, titre, col) in zip(axes, histories_data):
    ax.plot(hist['accuracy'],     color=col, label='Train')
    ax.plot(hist['val_accuracy'], color=col, label='Validation',
            linestyle='--', alpha=0.7)
    ax.set_title(titre, fontweight='bold')
    ax.set_xlabel('Époque')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

# ✅ Ligne verticale pour indiquer le début du fine-tuning
n_phase1 = len(history3_phase1.history['accuracy'])
axes[2].axvline(x=n_phase1, color='gray', linestyle=':', linewidth=1.5,
                label='Début fine-tuning')
axes[2].legend()

plt.suptitle("Courbes d'apprentissage", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('courbes_apprentissage.png', dpi=120)
plt.show()

---
## 8. Matrice de confusion — Meilleur modèle (Transfer Learning)

In [ ]:
# Prédictions avec le meilleur modèle
y_pred_prob = model3.predict(X_test)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = np.argmax(y_test,      axis=1)

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.title('Matrice de confusion — Transfer Learning (MobileNetV2)',
          fontsize=12, fontweight='bold')
plt.ylabel('Vraie catégorie')
plt.xlabel('Catégorie prédite')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('matrice_confusion.png', dpi=120)
plt.show()

In [ ]:
# Rapport détaillé par catégorie
print("Rapport de classification — Transfer Learning")
print("="*55)
print(classification_report(y_true, y_pred, target_names=CATEGORIES))

---
## 9. Test sur une image réelle

In [ ]:
def predire_image(chemin_image, modele):
    """Prend une image, retourne la catégorie prédite avec toutes les probabilités"""
    img = Image.open(chemin_image).convert('RGB').resize(IMG_SIZE)
    arr = np.array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)

    predictions = modele.predict(arr, verbose=0)[0]
    idx         = np.argmax(predictions)
    categorie   = CATEGORIES[idx]
    confiance   = predictions[idx] * 100

    # ✅ AJOUT : Afficher toutes les probabilités par classe
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(img)
    axes[0].set_title(f'Prédiction : {categorie} ({confiance:.1f}%)',
                      fontsize=12, fontweight='bold', color='darkgreen')
    axes[0].axis('off')

    colors = ['#2E8B57' if i == idx else '#AAAAAA' for i in range(NUM_CLASSES)]
    axes[1].barh(CATEGORIES, predictions * 100, color=colors)
    axes[1].set_xlabel('Probabilité (%)')
    axes[1].set_title('Probabilités par classe')
    axes[1].set_xlim(0, 100)

    plt.tight_layout()
    plt.show()

    print(f"Catégorie prédite : {categorie}")
    print(f"Confiance         : {confiance:.1f}%")
    return categorie

# Tester avec une image du dataset
exemple_path = os.path.join(BASE_DIR, CATEGORIES[0],
               os.listdir(os.path.join(BASE_DIR, CATEGORIES[0]))[0])
predire_image(exemple_path, model3)

---
## 10. Résumé final

In [ ]:
print("="*65)
print("RÉSUMÉ COMPARATIF DES 3 MODÈLES — VERSION AMÉLIORÉE")
print("="*65)
print(f"{'Modèle':<40} {'Accuracy':>10}")
print("-"*65)
print(f"{'1. CNN Simple':<40} {acc1*100:>9.1f}%")
print(f"{'2. CNN Profond + Data Augmentation':<40} {acc2*100:>9.1f}%")
print(f"{'3. Transfer Learning (MobileNetV2)':<40} {acc3*100:>9.1f}%")
print("="*65)
print()
print("✅ Améliorations appliquées :")
print("  - Résolution 224×224 (taille native MobileNetV2)")
print("  - Fine-tuning des 30 dernières couches (Phase 2)")
print("  - EarlyStopping + ReduceLROnPlateau + ModelCheckpoint")
print("  - Class weights (compensation déséquilibre 'trash')")
print("  - GlobalAveragePooling2D dans le CNN Profond")
print("  - LR du CNN Profond réduit à 5e-4")
print()
print("Modèles sauvegardés :")
print("  model_cnn_simple.h5")
print("  model_cnn_profond.h5")
print("  model_transfer_learning.h5")
print()
print("Meilleur modèle recommandé : Transfer Learning (MobileNetV2) avec fine-tuning")
print("=> Ce modèle sera utilisé pour le déploiement à l'Étape 4")